# Convert your data to imzML file format

This example will show you how you can convert your IMS data to imzML format using the imzy and pyimzML libraries.
The `imzy` library provides tools for working with imaging mass spectrometry data, while `pyimzML` is a library for reading and writing imzML files.

## Dependencies

To export your IMS data to imzML format, you will need the following libraries:

```bash
pip install imzy pyimzML "ims-utils>=0.1.8"
```

- imzy
- pyimzML
- ims-utils >=0.1.8  (added support for centroid -> profile conversion)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np

# this will handle writing imzML files
from pyimzml.ImzMLWriter import ImzMLWriter

# this will handle the supported file formats
from imzy import get_reader

In [3]:
path = Path(r"PATH-TO-YOUR-NEOFLEX_DATA")
reader = get_reader(path)
reader

In [ ]:
# for the Bruker neofleX and Waters readers, we provide automatic conversion from centroid to profile mode data
# this is carried out by creating a Gaussian peak shape for each peak in the centroided data
# and then sampling this peak shape at the m/z values of the profile data
# if you want to disable this feature, set auto_profile to False
reader.auto_profile = False


# create a list of coordinates
# need to add +1 to the coordinate since imimspy index starts at 0 and imzml expect it to start at 1
coordinates = np.c_[reader.x_coordinates, reader.y_coordinates] + 1

# get the data type of the first spectrum
# this is used to specify the data type when creating the imzML file
x, y = next(reader.spectra_iter(reader.pixels))


# specify the output path - in this case, we just want to save it to the same location the original data was located, with the .imzML extension
output_path = path.with_suffix(".imzML")
if output_path.exists():
    # if the file already exists, raise an error
    raise FileExistsError(f"The file {output_path} already exists. Please delete it or choose a different output path.")

# create an instance of the file reader
with (
    ImzMLWriter(
        output_path,
        mz_dtype=x.dtype.type,  # this can be adjusted depending on the data type of the m/z values (e.g. float32 or float64)
        intensity_dtype=y.dtype.type,  # this can be adjusted depending on the data type of the intensity values (e.g. int32 or float32)
        mode="processed",  # options are: continuous, processed or auto - continuous is profile mode (saves mass axis for first spectrum only), processed is centroided (saves mass axis for each spectrum)
        spec_type="centroid",  # options are: profile or centroid
        polarity="negative",  # options are: positive or negative
    ) as writer
):
    # iterate over all spectra in the file
    for i, (x, y) in enumerate(reader.spectra_iter(reader.pixels)):
        # get the coordinate values for this spectrum
        xy_coordinates = tuple(coordinates[i])
        # add the spectrum to the imzML file
        writer.addSpectrum(x, y, xy_coordinates)

Iterating spectra...: 100%|██████████| 3024/3024 [00:01<00:00, 2174.45it/s]


In [ ]:
# you can also just save the data in profile mode
# this will work for both profile and centroided data
reader.auto_profile = True

# specify the output path - in this case, we just want to save it to the same location the original data was located, with the .imzML extension
output_path = path.with_suffix("_profile.imzML")
if output_path.exists():
    # if the file already exists, raise an error
    raise FileExistsError(f"The file {output_path} already exists. Please delete it or choose a different output path.")

# create an instance of the file reader
with (
    ImzMLWriter(
        output_path,
        mz_dtype=np.float32,  # this can be adjusted depending on the data type of the m/z values (e.g. float32 or float64)
        intensity_dtype=np.float32,  # this can be adjusted depending on the data type of the intensity values (e.g. int32 or float32)
        mode="continuous",  # options are: continuous, processed or auto - continuous is profile mode (saves mass axis for first spectrum only), processed is centroided (saves mass axis for each spectrum)
        spec_type="profile",  # options are: profile or centroid
        polarity="negative",  # options are: positive or negative
    ) as writer
):
    # iterate over all spectra in the file
    for i, (x, y) in enumerate(reader.spectra_iter(reader.pixels)):
        # get the coordinate values for this spectrum
        xy_coordinates = tuple(coordinates[i])
        # add the spectrum to the imzML file
        writer.addSpectrum(x, y, xy_coordinates)